In [2]:
# =============================================================
#  STEP 1 — PREPROCESS + MERGE
#  Inputs:
#    - pune_s5p_clean_500m.csv     (2.5M rows, 500m satellite)
#    - ground_clean.csv            (2011 rows, 10 stations)
#    - road_density_pune.csv       (44,589 pixel lookup)
#
#  Outputs:
#    - data/output/satellite_500m_with_road.csv
#    - data/output/training_data_500m.csv
#    - data/output/preprocessing_report.json
#
#  Run: python3 step1_preprocess_and_merge.py
# =============================================================

import pandas as pd
import numpy as np
from scipy.spatial import KDTree
import os, json, warnings, time
warnings.filterwarnings("ignore")

os.makedirs("data/output", exist_ok=True)

SAT_PATH  = "data/satellite/pune_s5p_clean_full_500m.csv"
GND_PATH  = "data/ground_truth_data/ground_clean.csv"
ROAD_PATH = "data/auxiliary/road_density_pune.csv"

ROUND_D     = 5
MAX_DIST_KM = 1.0
MIN_READINGS = 8

STATION_COORDS = {
    "Bhumkar Nagar, Pune - IITM":                  (18.59800, 73.77300),
    "Gavalinagar_Pimpri_Chinchwad":                (18.63673, 73.82487),
    "Hadapsar, Pune - IITM":                       (18.50300, 73.93900),
    "Katraj_Dairy_Pune":                           (18.45445, 73.85416),
    "Panchawati_Pashan, Pune - IITM":              (18.53600, 73.82600),
    "Park Street Wakad, Pimpri Chinchwad - MPCB":  (18.59700, 73.76200),
    "Savitribai Phule Pune University, Pune - MPCB":(18.52900, 73.85100),
    "Savta Mali Nagar, Pimpri-Chinchwad - IITM":   (18.62800, 73.80600),
    "Thergaon, Pimpri Chinchwad - MPCB":           (18.61100, 73.78400),
    "Transport Nagar-Nigdi, Pune - IITM":          (18.65400, 73.78800),
}

report = {}


# =============================================================
#  A — SATELLITE
# =============================================================

def preprocess_satellite():
    print("\n" + "="*60)
    print("  A — SATELLITE 500m")
    print("="*60)
    print("  Loading 2.5M rows (30-60 sec)...")
    sat = pd.read_csv(SAT_PATH)
    print(f"  Loaded: {len(sat):,} rows  {sat.shape[1]} cols")

    sat["date"] = pd.to_datetime(sat["date"]).dt.strftime("%Y-%m-%d")

    # Drop redundant total NO2 (= trop + strat, causes multicollinearity)
    if "NO2_total_umol_m2" in sat.columns:
        sat = sat.drop(columns=["NO2_total_umol_m2"])
        print("  Dropped NO2_total_umol_m2 (redundant)")

    # Remove impossible NO2 values
    before = len(sat)
    sat = sat[(sat["NO2_trop_umol_m2"] > 0) & (sat["NO2_trop_umol_m2"] < 500)]
    sat = sat.dropna(subset=["latitude","longitude","NO2_trop_umol_m2"])
    print(f"  Removed {before - len(sat):,} invalid rows")

    # Round coords for clean joins
    sat["latitude"]  = sat["latitude"].round(ROUND_D)
    sat["longitude"] = sat["longitude"].round(ROUND_D)

    missing_aux = [c for c in ["elevation_m","population_density","land_surface_temp_c"]
                   if c not in sat.columns]

    report["satellite"] = {
        "rows_clean":    len(sat),
        "unique_pixels": int(sat[["latitude","longitude"]].drop_duplicates().shape[0]),
        "unique_dates":  int(sat["date"].nunique()),
        "date_range":    f"{sat['date'].min()} → {sat['date'].max()}",
        "missing_aux":   missing_aux,
    }

    print(f"  Clean: {len(sat):,} rows | {sat['date'].nunique()} dates | "
          f"{sat[['latitude','longitude']].drop_duplicates().shape[0]:,} pixels")
    if missing_aux:
        print(f"  ⚠️  Missing auxiliary features: {missing_aux}")
        print("      Run the new GEE script to include them")
    return sat


# =============================================================
#  B — ROAD DENSITY
# =============================================================

def preprocess_road():
    print("\n" + "="*60)
    print("  B — ROAD DENSITY")
    print("="*60)
    rd = pd.read_csv(ROAD_PATH)
    rd["latitude"]  = rd["latitude"].round(ROUND_D)
    rd["longitude"] = rd["longitude"].round(ROUND_D)
    rd = rd.drop_duplicates(subset=["latitude","longitude"])
    print(f"  Loaded: {len(rd):,} unique pixels")
    print(f"  road_weighted_norm range: {rd['road_weighted_norm'].min():.3f} – "
          f"{rd['road_weighted_norm'].max():.3f}")
    report["road_density"] = {"pixels": len(rd)}
    return rd


# =============================================================
#  C — MERGE ROAD → SATELLITE
# =============================================================

def merge_road(sat, rd):
    print("\n" + "="*60)
    print("  C — MERGE ROAD DENSITY INTO SATELLITE")
    print("="*60)

    road_cols = ["latitude","longitude","road_length_m","road_weighted_m",
                 "road_length_norm","road_weighted_norm"]

    # Direct join — same pixel grid (road has 44,589 pixels = satellite pixels)
    sat = sat.merge(rd[road_cols], on=["latitude","longitude"], how="left")

    matched   = sat["road_length_m"].notna().sum()
    unmatched = sat["road_length_m"].isna().sum()
    print(f"  Exact match: {matched:,} rows  ({matched/len(sat)*100:.2f}%)")

    if unmatched > 0:
        print(f"  Filling {unmatched:,} unmatched via KD-Tree nearest...")
        rd_tree  = KDTree(rd[["latitude","longitude"]].values)
        mask     = sat["road_length_m"].isna()
        _, idxs  = rd_tree.query(sat.loc[mask, ["latitude","longitude"]].values)
        for col in ["road_length_m","road_weighted_m",
                    "road_length_norm","road_weighted_norm"]:
            sat.loc[mask, col] = rd[col].values[idxs]

    print(f"  All road density values filled ✅")
    print(f"  Columns now: {sat.shape[1]}")
    return sat


# =============================================================
#  D — GROUND STATIONS
# =============================================================

def preprocess_ground():
    print("\n" + "="*60)
    print("  D — GROUND STATIONS")
    print("="*60)
    gnd = pd.read_csv(GND_PATH)
    gnd["date"] = pd.to_datetime(gnd["date"]).dt.strftime("%Y-%m-%d")

    # Inject precise coordinates
    gnd["latitude"]  = gnd["location_name"].map({k:v[0] for k,v in STATION_COORDS.items()}).fillna(gnd["latitude"])
    gnd["longitude"] = gnd["location_name"].map({k:v[1] for k,v in STATION_COORDS.items()}).fillna(gnd["longitude"])

    before = len(gnd)
    gnd = gnd[gnd["NO2_ground_ugm3"].between(1.0, 300.0)]
    gnd = gnd[gnd["reading_count"] >= MIN_READINGS]
    print(f"  Loaded: {before} rows → {len(gnd)} after cleaning")
    print(f"  NO2 range: {gnd['NO2_ground_ugm3'].min():.1f} – {gnd['NO2_ground_ugm3'].max():.1f} µg/m³")

    report["ground"] = {
        "rows": len(gnd), "stations": int(gnd["location_name"].nunique()),
        "date_range": f"{gnd['date'].min()} → {gnd['date'].max()}",
        "NO2_mean": round(float(gnd["NO2_ground_ugm3"].mean()), 2),
    }
    return gnd


# =============================================================
#  E — MERGE SATELLITE + GROUND → TRAINING DATA
# =============================================================

def merge_satellite_ground(sat, gnd):
    print("\n" + "="*60)
    print("  E — MERGE SATELLITE + GROUND → TRAINING DATA")
    print("="*60)

    sat_dates = set(sat["date"].unique())
    gnd_dates = set(gnd["date"].unique())
    overlap   = sat_dates & gnd_dates
    print(f"  Date overlap: {len(overlap)} days")

    sat_by_date = {d: grp.reset_index(drop=True) for d,grp in sat.groupby("date")}
    sat_feat_cols = [c for c in sat.columns if c != "date"]

    matched, skip_no_sat, skip_too_far = [], 0, 0

    for _, g in gnd.iterrows():
        date = g["date"]
        if date not in sat_by_date:
            skip_no_sat += 1
            continue
        sd      = sat_by_date[date]
        d, idx  = KDTree(sd[["latitude","longitude"]].values).query([g["latitude"], g["longitude"]])
        dist_km = d * 111.0
        if dist_km > MAX_DIST_KM:
            skip_too_far += 1
            continue
        pix = sd.iloc[idx]
        row = {
            "date": date, "station_name": g["location_name"],
            "match_dist_km": round(dist_km, 4),
            "gnd_lat": g["latitude"], "gnd_lon": g["longitude"],
            "NO2_ground_ugm3": round(g["NO2_ground_ugm3"], 4),
            "NO2_ground_std":  round(g["NO2_ground_std"],  4),
            "reading_count":   int(g["reading_count"]),
        }
        for col in sat_feat_cols:
            row[col] = pix[col]
        matched.append(row)

    df_train = pd.DataFrame(matched)
    print(f"  ✅ Matched: {len(df_train)}")
    print(f"  ❌ No satellite: {skip_no_sat}  |  Too far: {skip_too_far}")
    print(f"  Avg match dist: {df_train['match_dist_km'].mean():.4f} km")
    print(f"  Max match dist: {df_train['match_dist_km'].max():.4f} km")

    season_map = {0:"Winter",1:"Summer",2:"Monsoon",3:"Post-monsoon"}
    print(f"\n  Season distribution:")
    print(df_train["season"].map(season_map).value_counts().to_string())
    print(f"\n  Per-station counts:")
    print(df_train.groupby("station_name")["NO2_ground_ugm3"].agg(["count","mean"]).round(1).to_string())

    report["training"] = {
        "matched": len(df_train), "skip_no_sat": skip_no_sat,
        "skip_too_far": skip_too_far,
        "match_dist_mean_km": round(float(df_train["match_dist_km"].mean()), 4),
    }
    return df_train


# =============================================================
#  F — QUALITY CHECKS
# =============================================================

def quality_checks(sat, df_train):
    print("\n" + "="*60)
    print("  F — QUALITY CHECKS")
    print("="*60)

    checks = {
        "satellite_nulls":  int(sat.isnull().sum().sum()),
        "training_nulls":   int(df_train.isnull().sum().sum()),
        "road_in_satellite":"road_weighted_norm" in sat.columns,
        "road_in_training": "road_weighted_norm" in df_train.columns,
        "training_rows":    len(df_train),
        "all_4_seasons":    len(df_train["season"].unique()) == 4,
    }

    for k, v in checks.items():
        flag = "✅" if (v == 0 or v is True or (k == "training_rows" and v >= 500)) else "⚠️"
        print(f"  {flag}  {k}: {v}")

    report["checks"] = checks


# =============================================================
#  MAIN
# =============================================================

if __name__ == "__main__":
    t0 = time.time()
    print("█"*60)
    print("  STEP 1 — PREPROCESS + MERGE")
    print("█"*60)

    sat      = preprocess_satellite()
    rd       = preprocess_road()
    sat      = merge_road(sat, rd)

    # Save full satellite+road as prediction dataset
    sat_out = "data/output/satellite_500m_with_road.csv"
    sat.to_csv(sat_out, index=False)
    print(f"\n  Saved prediction dataset → {sat_out}")
    print(f"  ({len(sat):,} rows × {sat.shape[1]} cols)")

    gnd      = preprocess_ground()
    df_train = merge_satellite_ground(sat, gnd)

    train_out = "data/output/training_data_500m.csv"
    df_train.to_csv(train_out, index=False)
    print(f"\n  Saved training data → {train_out}")
    print(f"  ({len(df_train):,} rows × {df_train.shape[1]} cols)")

    quality_checks(sat, df_train)

    with open("data/output/preprocessing_report.json","w") as f:
        json.dump(report, f, indent=2)

    print(f"\n  Total time: {time.time()-t0:.0f}s")
    print("█"*60)
    print("  Next: python3 step2_train_and_predict.py")
    print("█"*60)


████████████████████████████████████████████████████████████
  STEP 1 — PREPROCESS + MERGE
████████████████████████████████████████████████████████████

  A — SATELLITE 500m
  Loading 2.5M rows (30-60 sec)...
  Loaded: 2,536,270 rows  18 cols
  Dropped NO2_total_umol_m2 (redundant)
  Removed 0 invalid rows
  Clean: 2,536,270 rows | 339 dates | 44,491 pixels

  B — ROAD DENSITY
  Loaded: 44,589 unique pixels
  road_weighted_norm range: 0.000 – 1.000

  C — MERGE ROAD DENSITY INTO SATELLITE
  Exact match: 2,536,270 rows  (100.00%)
  All road density values filled ✅
  Columns now: 21

  Saved prediction dataset → data/output/satellite_500m_with_road.csv
  (2,536,270 rows × 21 cols)

  D — GROUND STATIONS
  Loaded: 2011 rows → 2011 after cleaning
  NO2 range: 2.1 – 234.3 µg/m³

  E — MERGE SATELLITE + GROUND → TRAINING DATA
  Date overlap: 254 days
  ✅ Matched: 957
  ❌ No satellite: 657  |  Too far: 397
  Avg match dist: 0.2325 km
  Max match dist: 0.9405 km

  Season distribution:
season
